In [ ]:
import csv
from itertools import islice
import json
import os
from pathlib import Path
import time

import requests


In [ ]:
with open(r'agdm_data/agdm-Pilot-csv_simpleGeomSubset.csv', 'r', newline='', encoding='utf-8') as csv_file:
    r = csv.DictReader(csv_file)
    for row in islice(r, 5):
        print(f'{row["Short Title"]}: \t {row["Bounding Box"]} \t {row["Place Lookup"]}')

## Nominatim

Great API, but I think the usage policy will make it hard to tool around this.

In [ ]:
# Public Nominatim usage policy: https://operations.osmfoundation.org/policies/nominatim/
# No API key is required. Identify this application, stay below 1 request/second,
# use one thread, and cache results. Set NOMINATIM_USER_AGENT in your environment
# if you want to add a project URL or contact address.
NOMINATIM_BASE_URL = os.environ.get(
    'NOMINATIM_BASE_URL', 'https://nominatim.openstreetmap.org'
)
NOMINATIM_USER_AGENT = os.environ.get(
    'NOMINATIM_USER_AGENT', 'agdm-geometry-notebook/1.0 (academic data cleanup)'
)
CACHE_PATH = Path('agdm_data/nominatim_cache.json')
REQUEST_INTERVAL_SECONDS = 1.1

session = requests.Session()
session.headers.update({
    'User-Agent': NOMINATIM_USER_AGENT,
    'Accept': 'application/geo+json, application/json',
})

if CACHE_PATH.exists():
    with CACHE_PATH.open(encoding='utf-8') as cache_file:
        geocode_cache = json.load(cache_file)
else:
    geocode_cache = {}

last_request_at = 0.0

def geocode(place, limit=1):
    """Return a cached GeoJSON response for a place name."""
    global last_request_at

    place = place.strip()
    cache_key = json.dumps({'q': place, 'limit': limit}, sort_keys=True)
    if cache_key in geocode_cache:
        return geocode_cache[cache_key]

    wait = REQUEST_INTERVAL_SECONDS - (time.monotonic() - last_request_at)
    if wait > 0:
        time.sleep(wait)

    response = session.get(
        f'{NOMINATIM_BASE_URL}/search',
        params={'q': place, 'format': 'geojson', 'limit': limit},
        timeout=30,
    )
    last_request_at = time.monotonic()
    response.raise_for_status()
    result = response.json()

    geocode_cache[cache_key] = result
    CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    with CACHE_PATH.open('w', encoding='utf-8') as cache_file:
        json.dump(geocode_cache, cache_file, ensure_ascii=False, indent=2)

    return result

geometry = 'Milwaukee'
data = geocode(geometry)
data


## Who's On First (WOF) and Native DuckDB for geometry enhancement

What the enrichment tool would do
for records missing spatial metadata:

1. Read the existing geographic fields.
2. Use the most specific place already selected in OpenRefine:
   `City → County → Region → State/Province → Country → Continent.`
    - This is stored in the `Place Lookup` column.
3. Search WOF
4. Disambiguate using the broader geographic hierarchy???
5. Generate a proposed bounding box.
6. Convert it into Aardvark syntax:
`ENVELOPE(West,East,North,South)`
7. Record match confidence and provenance???
8. Produce a review file rather than silently updating authoritative metadata.
9. Generate Aardvark only from accepted results.

- load source records
- normalize place hierarchy
- generate candidates
- score candidates
- extract bounding box
- format Aardvark envelope
- validate coordinates
- export review report
- apply accepted matches

In [ ]:
import duckdb
import pandas as pd

# Optional, depending on geometry encoding:
import geopandas as gpd
from shapely import wkb, wkt

In [ ]:
!uv pip install geopandas shapely duckdb-engine duckdb

In [ ]:
# Block 1: Load the OpenRefine output and reduce it to unique lookup contexts.
# Broader fields are retained only to help distinguish places with the same name.
AGDM_CSV = Path('agdm_data/agdm-Pilot-csv_simpleGeomSubset.csv')
CONTEXT_COLUMNS = [
    'Place Lookup', 'City', 'County', 'Region',
    'State/Province', 'Country', 'Continent',
]

records = pd.read_csv(AGDM_CSV, dtype=str).fillna('')
missing_columns = set(CONTEXT_COLUMNS) - set(records.columns)
if missing_columns:
    raise ValueError(f'Missing expected columns: {sorted(missing_columns)}')

records['Place Lookup'] = records['Place Lookup'].str.strip()
lookups = (
    records.loc[records['Place Lookup'].ne(''), CONTEXT_COLUMNS]
    .drop_duplicates()
    .reset_index(drop=True)
)

print(f'{len(records):,} records; {len(lookups):,} unique lookup contexts')
lookups.head()

In [ ]:
# Block 2: Connect DuckDB to the official WOF administrative Parquet data.
# Set WOF_PARQUET to a local file path later if you download the dataset.
WOF_PARQUET = Path(
    "/mnt/d/wof/whosonfirst-data-admin-latest.parquet"
)

if not WOF_PARQUET.is_file():
    raise FileNotFoundError(f"WOF file not found: {WOF_PARQUET}")

print(f"Using local WOF data: {WOF_PARQUET}")
print(f"Size: {WOF_PARQUET.stat().st_size / 1024**3:.2f} GiB")

wof = duckdb.connect()
#wof.execute("SET enable_progress_bar = false")
wof_schema = wof.execute(
    'DESCRIBE SELECT * FROM read_parquet(?)',
    [str(WOF_PARQUET)],
).df()

wof_schema

In [ ]:
# Block 3: Return exact-name WOF candidates and resolve hierarchy IDs to names.
# This deliberately returns every match; choosing among them is a later review step.
def lookup_wof_candidates(place_name, limit=25):
    place_name = place_name.strip()
    if not place_name:
        return pd.DataFrame()

    return wof.execute(
        '''
        WITH candidates AS (
            SELECT
                id AS wof_id,
                name,
                placetype,
                country,
                parent_id,
                country_id,
                region_id,
                county_id,
                lat,
                lon,
                geometry_bbox
            FROM read_parquet(?)
            WHERE lower(name) = lower(?)
            ORDER BY placetype, name, id
            LIMIT ?
        ),
        ancestor_ids AS (
            SELECT DISTINCT unnest([
                cast(parent_id AS BIGINT),
                try_cast(nullif(country_id, '') AS BIGINT),
                try_cast(nullif(region_id, '') AS BIGINT),
                try_cast(nullif(county_id, '') AS BIGINT)
            ]) AS id
            FROM candidates
        ),
        ancestors AS (
            SELECT w.id, w.name, w.placetype
            FROM read_parquet(?) AS w
            INNER JOIN ancestor_ids AS a ON w.id = a.id
        )
        SELECT
            candidate.wof_id,
            candidate.name,
            candidate.placetype,
            candidate.country,
            candidate.parent_id,
            parent.name AS parent_name,
            parent.placetype AS parent_placetype,
            candidate.country_id,
            country.name AS country_name,
            candidate.region_id,
            region.name AS region_name,
            candidate.county_id,
            county.name AS county_name,
            candidate.lat,
            candidate.lon,
            candidate.geometry_bbox.xmin AS west,
            candidate.geometry_bbox.ymin AS south,
            candidate.geometry_bbox.xmax AS east,
            candidate.geometry_bbox.ymax AS north,
            CASE
                WHEN candidate.geometry_bbox IS NULL THEN NULL
                ELSE format(
                    'ENVELOPE({},{},{},{})',
                    candidate.geometry_bbox.xmin, candidate.geometry_bbox.xmax,
                    candidate.geometry_bbox.ymax, candidate.geometry_bbox.ymin
                )
            END AS aardvark_envelope
        FROM candidates AS candidate
        LEFT JOIN ancestors AS parent
            ON parent.id = candidate.parent_id
        LEFT JOIN ancestors AS country
            ON country.id = try_cast(nullif(candidate.country_id, '') AS BIGINT)
        LEFT JOIN ancestors AS region
            ON region.id = try_cast(nullif(candidate.region_id, '') AS BIGINT)
        LEFT JOIN ancestors AS county
            ON county.id = try_cast(nullif(candidate.county_id, '') AS BIGINT)
        ORDER BY candidate.placetype, candidate.name, candidate.wof_id
        ''',
        [str(WOF_PARQUET), place_name, limit, str(WOF_PARQUET)],
    ).df()

milwaukee_candidates = lookup_wof_candidates('Milwaukee')
milwaukee_candidates